# Solving the Heat Equation
- Python program to numerically solve a partial differential equation (PDE) called the "Heat Equation":

$ \frac{\partial u}{\partial t} = \alpha \frac{\partial^{2} u}{\partial x^{2}} $

- This PDE models the time evolution and spatial distribution of an initial temperature profile.
- $x$ is one spatial dimention, $t$ is time, $u$ is temperature and $\alpha$ is thermal diffusivity of the medium.
- Solution method: FTCS method (Forward difference in Time; Central difference in Space)
- The particular solution of this PDE is visualised in the form of an animation.

In [ ]:
"""
@author: Apoorva Upadhyay
"""
"""
- Solve heat equation. u = temperature of metal bar
- Use FTCS (forward in time, central in space) method
- Animate particular solution. See HeatEquationSolution_animation.mp4
"""
from numpy import *
from matplotlib.pyplot import *
import numpy as np
import decimal
from decimal import Decimal

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi']=300 # highres display

from mpl_toolkits.mplot3d import Axes3D # import 3d plotting tool

import os

import re

import pandas as pd

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
plt.rcParams['text.usetex'] = True

import math
import statistics

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation

from matplotlib.animation import PillowWriter
from matplotlib.animation import FFMpegFileWriter
from matplotlib.animation import FFMpegWriter


#######################################################################

# dicretise x axis
L = 6
a = 1

x_min = -L/2
x_max = L/2
delta_x = 0.1
N = int((x_max - x_min)/(delta_x)+1)

x = np.linspace(x_min,x_max,N)

# dicretise time t "axis"
t_min = 0.0
t_max = 1.1
delta_t = 0.001

# gaussian profile
def u_x_t0(x):
    
    sigma_vals = 1.0
    mu_vals = 1.0
    u_x_t0 = (1/sigma_vals*np.sqrt(2*np.pi)) * np.exp(-(x-mu_vals)**2 / 2*sigma_vals**2)
    
    return u_x_t0

    
u_x_t = np.zeros(N)
    
t = 0.0

def Tenperature(u_x_t0_init, a, delta_t, delta_x):
    
    # index j runs over spatial points on x axis
    for j in range(0,len(u_x_t0_init)-1,1):
        
        u_x_t[j] = u_x_t0_init[j] + ( (((a**2)*delta_t)/(delta_x**2)) * (u_x_t0_init[j+1] + u_x_t0_init[j-1] - 2*u_x_t0_init[j]) )
        
        u_x_t[0] = 0.0 
        u_x_t[-1] = 0.0 
        
    return u_x_t


writer = FFMpegWriter(fps=15)

x_vals = []
y_vals = []

fig, ax1 = plt.subplots(1,1, figsize=(8,4))
#l, = plt.plot([],[], 'r-')
l, = ax1.plot([],[], 'r-')

time_text = ax1.text(0.65, 0.95, '', fontsize=15, transform=ax1.transAxes, bbox=dict(facecolor='white',edgecolor='black'))

ax1.set_xlim(-3,3)
ax1.set_ylim(0,5)


ax2 = inset_axes(ax1, width="80%", height="10%", loc=3, borderpad=1, bbox_to_anchor=(.0, 0.7, 1.0, 0.97), bbox_transform=ax1.transAxes)
ax2.set_xlabel(r'',size=4)
ax2.set_ylabel(r'',size=4)
ax2.tick_params(axis='both', which='major', labelsize=5)



u_x_t0_init = u_x_t0(x) # Initial Condition: initial gaussian profile

#plt.plot(x, u_x_t0_init)

# Boundary Conditions:
u_x_t0_init[0] = 0.0
u_x_t0_init[-1] = 0.0

ax1.plot(x, u_x_t0_init, 'b--')

time = np.arange(t_min,t_max,delta_t)    

ax2.imshow(np.atleast_2d(u_x_t0_init), cmap=plt.get_cmap('coolwarm'), extent=(-3, 3, 0, 1))

with writer.saving(fig, 'HeatEquationSolution_animation.mp4', 100):
    
    for k in range(0,len(time)-1,1):
        
        u_x_t = Tenperature(u_x_t0_init,a,delta_t,delta_x)
        
        
        x_vals.append(x)
        y_vals.append(u_x_t)
        
        l.set_data(x_vals[k], y_vals[k]) 
        time_text.set_text('t={:.2f}'.format(time[k]))
        
        ax2.imshow(np.atleast_2d(y_vals[k]), cmap=plt.get_cmap('coolwarm'), extent=(-3, 3, 0, 1))
        
        writer.grab_frame()
        
        u_x_t0_init = u_x_t 
        
        